## Leetcode 177. Nth Highest Salary

In [0]:
data = [(1,100),(2,200),(3,300)]
Employee = spark.createDataFrame(data, ['id','salary'])
Employee.createOrReplaceTempView('Employee')
N = 2


Employee
| id | salary |
|----|--------|
| 1  | 100    |
| 2  | 200    |
| 3  | 300    |

### Spark SQL

In [0]:
spark.sql(f'''
          select coalesce((select distinct salary 
          from (select salary, dense_rank() over(order by salary desc) rnk from Employee)
          where rnk = {N}), null) as `getNthHighestSalary({N})`
          ''').show()

### Spark DataFrame API

In [0]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

In [0]:
from pyspark.sql.functions import col, dense_rank, lit, coalesce
from pyspark.sql.window import Window

spark.conf.set("spark.sql.shuffle.partitions", "1")
N=2
w = Window.orderBy(col('salary').desc())

result = Employee.withColumn('rnk', dense_rank().over(w))\
    .filter(col('rnk') == N)\
    .select(col('salary'))\
    .withColumnRenamed('salary', f'getNthHighestSalary({N})')

result = result.select(coalesce(col(f'getNthHighestSalary({N})'), lit(None)).alias(f'getNthHighestSalary({N})'))

result.show()